In [1]:
# --- v6 Clean PoC Bootstrap ---
import os, sys
from pathlib import Path

if "/mnt/data" not in sys.path:
    sys.path.insert(0, "/mnt/data")

DATA_ROOT = os.environ.get("DATA_ROOT", "/mnt/data")
CONFIG = {
    "DATA_ROOT": DATA_ROOT,
    "EQUIPMENT_STATUS_PATH": str(Path(DATA_ROOT) / "equipment_status.csv"),
    "EQUIPMENT_MOVES_LOG_PATH": str(Path(DATA_ROOT) / "equipment_moves.csv"),
    "SOP_REGISTRY_PATH": str(Path(DATA_ROOT) / "sop_registry.csv"),
    "QR_OUTPUT_DIR": str(Path(DATA_ROOT) / "qr"),
    "EVENT_LOG_PATH": str(Path(DATA_ROOT) / "event_log.jsonl"),
    "RUN_UI": False,
    "RUN_PIPELINE": False,
}

# ensure dirs
for k in ["QR_OUTPUT_DIR","EVENT_LOG_PATH","SOP_REGISTRY_PATH","EQUIPMENT_STATUS_PATH","EQUIPMENT_MOVES_LOG_PATH"]:
    p = Path(CONFIG[k])
    (p.parent if p.suffix else p).mkdir(parents=True, exist_ok=True)

RUN_UI = CONFIG["RUN_UI"]
RUN_PIPELINE = CONFIG["RUN_PIPELINE"]

print("v6 clean bootstrap ready")

v6 clean bootstrap ready


In [2]:
from typing import Optional, Dict, Any, List, Tuple
from __future__ import annotations
from dataclasses import dataclass, field

@dataclass
class WorkflowState:
    encounter_id: Optional[str] = None
    patient_id: Optional[str] = None
    pending_orders: set = field(default_factory=set)
    completed_studies: set = field(default_factory=set)
    active_consults: set = field(default_factory=set)
    last_vitals_ts: Optional[pd.Timestamp] = None
    chest_pain: bool = False
    trauma: bool = False
    # context
    backlog_ct: int = 0
    backlog_lab: int = 0
    backlog_ecg: int = 0
    hour: int = 12
    role: str = "nurse"

def skill_need_ecg(state: WorkflowState) -> Optional[Dict[str,Any]]:
    if state.chest_pain and ("ORDER_ECG" not in state.pending_orders) and ("ORDER_ECG" not in state.completed_studies):
        return {"action":"ORDER_ECG", "reason":"Chest pain without ECG", "urgency":"high"}
    return None

def skill_abnormal_ecg_no_consult(state: WorkflowState) -> Optional[Dict[str,Any]]:
    if ("ORDER_ECG" in state.completed_studies) and ("ECG_ABNORMAL" in state.completed_studies) and ("CARDIOLOGY" not in state.active_consults):
        return {"action":"PAGE_CARDIOLOGY", "reason":"Abnormal ECG without consult", "urgency":"high"}
    return None

def skill_ct_delayed(state: WorkflowState) -> Optional[Dict[str,Any]]:
    if ("ORDER_CT" in state.pending_orders) and ("CT_RESULT" not in state.completed_studies):
        return {"action":"FOLLOW_UP_IMAGING", "reason":"CT pending > 60m", "urgency":"medium"}
    return None

def skill_pending_labs_deteriorating(state: WorkflowState) -> Optional[Dict[str,Any]]:
    if (("LAB_TROPONIN" in state.pending_orders) or ("LAB_PANEL" in state.pending_orders)) and ("Deteriorating" in state.completed_studies):
        return {"action":"EXPEDITE_LABS", "reason":"Pending labs + deterioration", "urgency":"high"}
    return None

SKILLS = [
    skill_need_ecg,
    skill_abnormal_ecg_no_consult,
    skill_ct_delayed,
    skill_pending_labs_deteriorating,
]

def generate_candidates(state: WorkflowState) -> List[Dict[str,Any]]:
    out = []
    for s in SKILLS:
        r = s(state)
        if r: out.append(r)
    return out[:5]


In [3]:
# --- TinyCritics (v6 patched) ---
from __future__ import annotations
from typing import List, Dict, Any, Tuple
import numpy as np, pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.calibration import CalibratedClassifierCV

class TinyCritics:
    def __init__(self):
        base = Pipeline([("impute", SimpleImputer(strategy="most_frequent")),("clf", LogisticRegression(max_iter=1000))])
        self.model = CalibratedClassifierCV(base, method="isotonic", cv=3)
        self.num_features_: List[str] = ["hour","spo2","backlog_ct","backlog_lab","backlog_ecg","pending_n","completed_n","consults_n","since_vitals_min"]
        self.cat_features_: List[str] = ["role","cp","resp","trauma"]
        self.preproc = ColumnTransformer([("num", SimpleImputer(strategy="median"), self.num_features_),("cat", OneHotEncoder(handle_unknown="ignore"), self.cat_features_)], remainder="drop")
        self.is_fit = False
    def _featurize(self, X: List[Dict[str,Any]]) -> pd.DataFrame:
        rows = []
        for x in X:
            s = x.get("state"); a = x.get("action", {})
            if hasattr(s, "feature_dict"): f = s.feature_dict()
            elif isinstance(s, dict): f = dict(s)
            else: f = {}
            f["action_label"] = str(a.get("label") or a.get("id") or "action")
            rows.append(f)
        df = pd.DataFrame(rows)
        for col in self.num_features_ + self.cat_features_:
            if col not in df.columns: df[col] = np.nan if col in self.num_features_ else "NA"
        return df[self.num_features_ + self.cat_features_ + ["action_label"]]
    def fit(self, samples: List[Dict[str,Any]], y: np.ndarray) -> "TinyCritics":
        df = self._featurize(samples)
        Xp = self.preproc.fit_transform(df[self.num_features_ + self.cat_features_]); self.model.fit(Xp, y); self.is_fit = True; return self
    def score(self, state, actions: List[Dict[str,Any]]):
        X = self._featurize([{"state": state, "action": a} for a in actions])
        if not self.is_fit:
            n = len(actions); return np.full(n, 0.5), np.zeros(n), np.zeros(n)
        Xp = self.preproc.transform(X[self.num_features_ + self.cat_features_])
        p = self.model.predict_proba(Xp)[:, 1]
        benefit = (1.0 - np.clip(X["backlog_ct"].fillna(0), 0, 10)/10.0).to_numpy()
        burden = (np.clip(X["since_vitals_min"].fillna(60), 0, 120)/120.0).to_numpy()
        return p, benefit, burden


In [4]:
# --- Phase2/3 bridge (optional; guarded) ---
import phase2_bridge as p2

def feature_dict_with_phase2(state):
    """Augment WorkflowState.feature_dict() with ICU flags if available."""
    base = state.feature_dict() if hasattr(state, "feature_dict") else {}
    extra = p2.run_icu_constraints(state) if RUN_PIPELINE else {}
    # In case of collisions, prefer base
    return {**extra, **base}

def routed_actions(state, actions):
    """Route actions via agent mesh if available; identity otherwise."""
    return p2.mesh_route_actions(state, actions) if RUN_PIPELINE else actions

def maybe_fit_with_trainer(critic, samples, y):
    """Use external trainer if available; else leave critic as-is."""
    return p2.trainer_fit_critic(critic, samples, y) if RUN_PIPELINE else critic

print("Phase2/3 bridge ready (active only when RUN_PIPELINE=True and modules are available).")


Phase2/3 bridge ready (active only when RUN_PIPELINE=True and modules are available).


In [5]:
# --- TinyCritics smoke test ---
import numpy as np, pandas as pd
st = WorkflowState(role="nurse")
if hasattr(st, "touch_now"): st.touch_now(pd.Timestamp.utcnow())
for k,v in {"cp":True,"resp":False,"trauma":False,"backlog_ct":1,"backlog_lab":2,"backlog_ecg":0,"spo2":94.0}.items():
    if hasattr(st,k): setattr(st,k,v)
tc = TinyCritics()
# Optional trainer path (no-op unless RUN_PIPELINE and trainer available)
tc = maybe_fit_with_trainer(tc, [], [])
# Route actions optionally
acts =  routed_actions(st, [{"id":"reassess_vitals","label":"Reassess vitals"},{"id":"order_ecg","label":"Order ECG"}])
p,b,u = tc.score(st, acts)
print("cold-start:", p.round(3), b.round(3), u.round(3))

cold-start: [0.5 0.5] [0. 0.] [0. 0.]


In [6]:
# --- Core tracker integration ---
from tracker_core import TrackerService
import tracker_ui
import pandas as pd
tracker = TrackerService.from_config(CONFIG)

def get_state():
    s = WorkflowState(role="nurse")
    if hasattr(s,"touch_now"): s.touch_now(pd.Timestamp.utcnow()); return s
def get_actions(s):
    return routed_actions(s, [{"id":"reassess_vitals","label":"Reassess vitals"},{"id":"order_ecg","label":"Order ECG"}])

if RUN_UI:
    tracker_ui.run_ui(tracker=tracker, get_state=get_state, get_actions=get_actions, critic=TinyCritics())
else:
    print("Tracker ready. Set CONFIG['RUN_UI']=True to launch UI.")

Tracker ready. Set CONFIG['RUN_UI']=True to launch UI.


In [7]:
# --- Sanity Pack ---
from tracker_core import QRService, SOPRegistry, EquipmentRepository, MovesLogRepository
qrs = QRService(CONFIG["QR_OUTPUT_DIR"]); out = qrs.make("v6-sanity"); print("QR:", out)
sop = SOPRegistry(CONFIG["SOP_REGISTRY_PATH"]).read(); print("SOP rows:", len(sop))
eq = EquipmentRepository(CONFIG["EQUIPMENT_STATUS_PATH"]).read(); print("Equip rows:", len(eq))
MovesLogRepository(CONFIG["EQUIPMENT_MOVES_LOG_PATH"]); print("Moves log ok")

QR: /mnt/data/qr/qr_3455791871756297792.png
SOP rows: 0
Equip rows: 0
Moves log ok


In [8]:
# --- Requirements report (non-failing) ---
from pathlib import Path
try:
    from PyPDF2 import PdfReader
    paths = [
        "/mnt/data/ED Research System - Priority Implementation Checklist.pdf",
        "/mnt/data/ED Workflow Clinical Requirements - Complete Specifications.pdf"
    ]
    cnt = 0
    for p in paths:
        if Path(p).exists():
            r = PdfReader(p); cnt += len(r.pages)
    print("PDFs available; pages:", cnt)
except Exception as e:
    print("PDF parse skipped:", e)

PDFs available; pages: 10
